# MOTCO End-to-End Example

This notebook walks through the complete MOTCO workflow using the bundled `evo_649_sm_example1.csv` dataset.

**Dataset:** 180 samples, 2 features (V1, V2), group column `taxa`, level column `Inv`.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.decomposition import PCA

from motco.stats.sd import (
    get_model_matrix, build_ls_means, estimate_difference,
    RRPP, get_observed_vectors, center_matrix,
)

# Load the bundled example dataset
data_path = Path('../tests/data/evo_649_sm_example1.csv')
df = pd.read_csv(data_path)
print(df.head())
print(f'Shape: {df.shape}, Groups: {df["taxa"].unique()}, Levels: {df["Inv"].unique()}')

## Step 1 — Define column roles

- `group_col`: the between-subject factor (which taxon / group the sample belongs to)
- `level_col`: the within-group factor (which timepoint / state the sample is at)
- Feature columns: all remaining numeric columns — these are the outcome `Y`

In [ ]:
group_col = 'taxa'
level_col = 'Inv'
feat_cols = [c for c in df.select_dtypes(include=[float]).columns if c not in {group_col, level_col}]

# Reduce to 2 PCA dimensions as the outcome space
pca = PCA(n_components=2, random_state=0)
Y = pd.DataFrame(pca.fit_transform(df[feat_cols]), columns=['PC1', 'PC2'])
print(f'Y shape: {Y.shape}')
print(f'Variance explained: {pca.explained_variance_ratio_}')

## Step 2 — Build the model matrix and LS means

`get_model_matrix` creates a design matrix with:
- An intercept column
- Dummy variables for groups (drop-first coding)
- Dummy variables for levels (drop-first coding)
- Group × level interaction terms (when `full=True`)

`build_ls_means` creates one row per group × level cell, in the same column coding. These rows, multiplied by the regression coefficients, give the least-squares mean vector for each cell.

In [ ]:
factors = df[[group_col, level_col]].copy()
g_levels = sorted(df[group_col].astype(str).unique())
l_levels = sorted(df[level_col].astype(str).unique())
print(f'Groups: {g_levels}')
print(f'Levels: {l_levels}')

M_full = get_model_matrix(factors, group_col=group_col, level_col=level_col, full=True)
M_red  = get_model_matrix(factors, group_col=group_col, level_col=level_col, full=False)
LS = build_ls_means(g_levels, l_levels, full=True)

print(f'Full model matrix shape: {M_full.shape}')
print(f'Reduced model matrix shape: {M_red.shape}')
print(f'LS means shape: {LS.shape}  (one row per group×level cell)')

## Step 3 — Inspect LS-mean vectors

`get_observed_vectors` fits the full model and returns the predicted mean position (in Y space) for each group × level combination.

In [ ]:
obs = get_observed_vectors(factors, Y, group_col=group_col, level_col=level_col, full=True)
print(obs)

## Step 4 — Define the contrast

The `contrast` list tells `estimate_difference` which LS-mean rows belong to each group's trajectory.

Row order in `LS` is group-major, level-minor:
- Row 0: group[0], level[0]
- Row 1: group[0], level[1]
- Row 2: group[1], level[0]
- Row 3: group[1], level[1]
- …

In [ ]:
L = len(l_levels)
contrast = [[gi * L + li for li in range(L)] for gi in range(len(g_levels))]
print('Contrast:', contrast)
print('Meaning:')
for gi, group in enumerate(g_levels):
    print(f'  Group {group}: LS-mean rows {contrast[gi]} → levels {l_levels}')

## Step 5 — Estimate trajectory differences

`estimate_difference` returns three symmetric matrices comparing every pair of groups:
- `deltas`: difference in trajectory magnitude (total path length)
- `angles`: angle in degrees between trajectory directions (0°=same, 90°=orthogonal, 180°=opposite)
- `shapes`: Procrustes distance between trajectory shapes

In [ ]:
deltas, angles, shapes = estimate_difference(Y, M_full, LS, contrast)
group_labels = g_levels
print('Angles (degrees):')
print(pd.DataFrame(angles, index=group_labels, columns=group_labels).round(2))
print('\nDeltas (magnitude difference):')
print(pd.DataFrame(deltas, index=group_labels, columns=group_labels).round(4))

## Step 6 — RRPP permutation test

RRPP permutes the residuals of the reduced model to build a null distribution for each statistic. The p-value is right-tailed with the add-one correction.

**Note:** Set `MOTCO_NOTEBOOK_PERMS=999` or higher for a real analysis. We use 99 here for speed.

In [ ]:
import os
PERMS = int(os.getenv('MOTCO_NOTEBOOK_PERMS', '99'))

dist_delta, dist_angle, _ = RRPP(Y, M_full, M_red, LS, contrast, permutations=PERMS)

def pvalue(samples, observed, i, j):
    vals = np.array([s[i, j] for s in samples])
    return (np.sum(vals >= observed) + 1) / (len(vals) + 1)

print(f'Results ({PERMS} permutations):')
for i, g1 in enumerate(g_levels):
    for j, g2 in enumerate(g_levels):
        if j <= i:
            continue
        ang = angles[i, j]
        dlt = deltas[i, j]
        p_ang = pvalue(dist_angle, ang, i, j)
        p_dlt = pvalue(dist_delta, dlt, i, j)
        print(f'  {g1} vs {g2}: angle={ang:.2f}° (p={p_ang:.3f}), delta={dlt:.4f} (p={p_dlt:.3f})')

## Equivalent CLI commands

The analysis above maps to:

```bash
# Estimate differences
motco de \\
  --Y Y.csv \\
  --model-matrix model_full.csv \\
  --ls-means ls_means.csv \\
  --contrast contrast.json \\
  --out-json result.json \\
  --out-observed ls_mean_vectors.csv

# With RRPP
motco de \\
  --Y Y.csv \\
  --model-full model_full.csv \\
  --model-reduced model_reduced.csv \\
  --ls-means ls_means.csv \\
  --contrast contrast.json \\
  --rrpp-permutations 999 \\
  --out-json rrpp_result.json
```